In [3]:
import math
import random
import yaml
import argparse
from dotmap import DotMap

import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam

# import matplotlib.pyplot as plt
import wandb

import sys
sys.path.append("../src")  # make sure Python can find src/
import data
from model import GPTLinear, GPTSoftmax
from multi_task_train import train_step


def set_seed(seed: int):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    ## Not sure if below would work if I dont have gpu
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Seed set to {seed}")


def load_config(config_path: str):
    """Load YAML config and convert to DotMap."""
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    cfg = DotMap(cfg)
    return cfg


def prepare_data_samplers(config, device):
    """Create a dict of data samplers for each task."""
    num_task = len(config.data.tasks)
    data_samplers = {}
    for task_config in config.data.tasks:
        task_name = task_config.name
        task_class = getattr(data, task_name)
        data_samplers[task_name] = {
            "sampler": task_class(task_config, device),
            "n_train": task_config.n_train,
            "n_test": task_config.n_test,
        }
    return data_samplers

In [17]:
config_prefix = load_config('../src/configs/prefixsum.yaml')
config_prefix.data.tasks[0].sep = 18
config_mws = load_config('../src/configs/mws.yaml')

In [18]:
config_prefix

DotMap(model=DotMap(n_layer=1, n_head=1, n_embd=256, linear=True, vocab_size=20), data=DotMap(tasks=[DotMap(name='PrefixSum', sep=18, n_train=128, n_test=64, min_num=1, max_num=16, p=17)], cot=False, num_tokens=16, n_train=256, n_test=64, fixed_len=True, mix='random'), train=DotMap(lr=0.0001, grad_clip=-1, num_steps=1000, norm_type='none_rank', wandb=True, wandb_run_name='mws_linear_embeddingNotFrozen', wandb_project='loss_plateau_tf', save_ckpt=False, ckpt_freq=20, seed=67, mask_input=True, freeze_embedding=False), _ipython_display_=DotMap(), _repr_mimebundle_=DotMap())

In [16]:
config_mws

DotMap(model=DotMap(n_layer=1, n_head=1, n_embd=256, linear=True, vocab_size=20), data=DotMap(tasks=[DotMap(name='MovingWindowSum', sep=17, n_train=128, n_test=64, min_num=1, max_num=16, k=2, p=17)], min_num=1, max_num=16, k=2, p=17, cot=False, num_tokens=16, n_train=256, n_test=64, fixed_len=True, mix='random'), train=DotMap(lr=0.0001, grad_clip=-1, num_steps=1000, norm_type='none_rank', wandb=True, wandb_run_name='mws_linear_embeddingNotFrozen', wandb_project='loss_plateau_tf', save_ckpt=False, ckpt_freq=20, seed=67, mask_input=True, freeze_embedding=False), _ipython_display_=DotMap(), _repr_mimebundle_=DotMap())

In [21]:
device = "cuda" if torch.cuda.is_available() else "cpu"

seed = getattr(config_prefix.train, "seed", 42)
set_seed(seed)

Seed set to 67


In [24]:
for config in [config_prefix, config_mws]:
    # config.model.vocab_size = max(getattr(config.data, "p", 17), config.data.max_num) + n_tasks
    config.model.block_size = 2 * config.data.num_tokens + 1

In [28]:
data_samplers_prefix = prepare_data_samplers(config_prefix, device)
data_samplers_prefix

{'PrefixSum': {'sampler': <data.prefix_sum.PrefixSum at 0x7f13284d3670>,
  'n_train': 128,
  'n_test': 64}}

In [29]:
data_samplers_mws = prepare_data_samplers(config_mws, device)
data_samplers_mws

{'MovingWindowSum': {'sampler': <data.moving_window_sum.MovingWindowSum at 0x7f13283c7e50>,
  'n_train': 128,
  'n_test': 64}}

In [33]:
if config.model.linear:
        model = GPTLinear(config.model, return_att=True).to(device)
else:
    model = GPTSoftmax(config.model, return_att=True).to(device)


In [36]:
if config.train.freeze_embedding: ### NEed to double check
        for param in model.transformer.wte.parameters():
            param.requires_grad = False
        for param in model.transformer.wpe.parameters():
            param.requires_grad = False

In [41]:
checkpoint = torch.load("ckpt_after_prefix.pt", map_location="cpu")

model.load_state_dict(checkpoint["model"])

<All keys matched successfully>

In [ ]:
### Freeze MLP layer 

In [42]:
model

GPTLinear(
  (transformer): ModuleDict(
    (wte): Embedding(20, 256)
    (wpe): Embedding(33, 256)
    (h): ModuleList(
      (0): Block(
        (ln_1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (k): Linear(in_features=256, out_features=256, bias=True)
          (q): Linear(in_features=256, out_features=256, bias=True)
          (v): Linear(in_features=256, out_features=256, bias=True)
          (c_proj): Linear(in_features=256, out_features=256, bias=True)
        )
        (ln_2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (mlp): ModuleDict(
          (c_fc): Linear(in_features=256, out_features=1024, bias=True)
          (c_proj): Linear(in_features=1024, out_features=256, bias=True)
          (act): NewGELU()
        )
      )
    )
    (ln_f): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=256, out_features=20, bias=False)
)

In [45]:
### Freezing everything but the attention:
# Freeze MLP
for block in model.transformer.h:
    for param in block.mlp.parameters():
        param.requires_grad = False

# Freeze final layer norm
for param in model.transformer.ln_f.parameters():
    param.requires_grad = False

# Freeze output head
for param in model.lm_head.parameters():
    param.requires_grad = False

In [49]:
stop_on_perfect = getattr(config_mws.train, "stop_on_perfect_acc", True)
stop_on_perfect = True


In [50]:
# Optimizer
optim = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=config.train.lr)

    # WandB setup
if getattr(config_mws.train, "wandb", False):
    wandb_run_name = "prefix_to_mws_freeze_MLP_and_output"
    wandb.login(key="")
    wandb.init(project=config.train.wandb_project, name=wandb_run_name, config=config, save_code=False)
    if getattr(config.train, "wandb_save_model", False):
        wandb.watch(model, log="parameters")  # save model parameters
    else:   
        wandb.watch(model, log=None)        # metrics only, no model saved

stop_on_perfect = True
perfect_patience = 50
# acc_eps = getattr(config.train, "perfect_acc_eps", 1e-6)

perfect_counter = 0

# Training loop
for step in range(config.train.num_steps):
    overall_metrics = train_step(
        model=model,
        optim=optim,
        data_samplers=data_samplers_mws,
        step=step,
        config=config,
        device=device,
    )
    
    ## Early stop
    if stop_on_perfect:
        acc = overall_metrics["test_acc"]  # or "train_acc" if you prefer

        if acc >= 1.0:
            perfect_counter += 1
        else:
            perfect_counter = 0

        if perfect_counter >= perfect_patience:
            print(
                f"\n✅ Early stopping at step {step}: "
                f"overall accuracy stayed at 100% for "
                f"{perfect_patience} consecutive steps\n"
            )
            break

if getattr(config.train, "wandb", False):
    wandb.finish()

/home/jyue/.conda/envs/emerge/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: jetyue04 (wth_ucsd). Use `wandb login --relogin` to force relogin


Step 0 -- Train loss: 5.936969757080078, Train Acc: 0.1748046875 Test Acc: 0.1728515625
Step 1 -- Train loss: 5.460556507110596, Train Acc: 0.1748046875 Test Acc: 0.169921875
Step 2 -- Train loss: 4.868325233459473, Train Acc: 0.17626953125 Test Acc: 0.1640625
Step 3 -- Train loss: 4.285306453704834, Train Acc: 0.1767578125 Test Acc: 0.177734375
Step 4 -- Train loss: 3.6279077529907227, Train Acc: 0.17578125 Test Acc: 0.1787109375
Step 5 -- Train loss: 3.091134548187256, Train Acc: 0.193359375 Test Acc: 0.1953125
Step 6 -- Train loss: 2.6075944900512695, Train Acc: 0.22998046875 Test Acc: 0.2314453125
Step 7 -- Train loss: 2.296236753463745, Train Acc: 0.28173828125 Test Acc: 0.279296875
Step 8 -- Train loss: 2.019158363342285, Train Acc: 0.3525390625 Test Acc: 0.3447265625
Step 9 -- Train loss: 1.814475178718567, Train Acc: 0.3896484375 Test Acc: 0.4013671875
Step 10 -- Train loss: 1.6358784437179565, Train Acc: 0.470703125 Test Acc: 0.46484375
Step 11 -- Train loss: 1.441609621047973

wandb: 429 encountered (Filestream rate limit exceeded, retrying in 2.4 seconds.), retrying request


Step 15 -- Train loss: 0.9645393490791321, Train Acc: 0.75390625 Test Acc: 0.763671875
Step 16 -- Train loss: 0.8429824113845825, Train Acc: 0.80078125 Test Acc: 0.8154296875
Step 17 -- Train loss: 0.7469782829284668, Train Acc: 0.845703125 Test Acc: 0.8369140625
Step 18 -- Train loss: 0.7009518146514893, Train Acc: 0.8564453125 Test Acc: 0.875
Step 19 -- Train loss: 0.6432892680168152, Train Acc: 0.87548828125 Test Acc: 0.8935546875
Step 20 -- Train loss: 0.5201384425163269, Train Acc: 0.92578125 Test Acc: 0.8984375
Step 21 -- Train loss: 0.5129812359809875, Train Acc: 0.9208984375 Test Acc: 0.9345703125
Step 22 -- Train loss: 0.4625553488731384, Train Acc: 0.927734375 Test Acc: 0.9296875
Step 23 -- Train loss: 0.4338405728340149, Train Acc: 0.93310546875 Test Acc: 0.94140625


wandb: 429 encountered (Filestream rate limit exceeded, retrying in 2.0 seconds.), retrying request
wandb: 429 encountered (Filestream rate limit exceeded, retrying in 4.4 seconds.), retrying request


Step 24 -- Train loss: 0.3792986273765564, Train Acc: 0.94482421875 Test Acc: 0.927734375
Step 25 -- Train loss: 0.41327089071273804, Train Acc: 0.9326171875 Test Acc: 0.93359375
Step 26 -- Train loss: 0.39553219079971313, Train Acc: 0.9365234375 Test Acc: 0.9482421875
Step 27 -- Train loss: 0.35138440132141113, Train Acc: 0.9453125 Test Acc: 0.9404296875
Step 28 -- Train loss: 0.3646690845489502, Train Acc: 0.94140625 Test Acc: 0.9423828125
Step 29 -- Train loss: 0.35413461923599243, Train Acc: 0.94140625 Test Acc: 0.935546875
Step 30 -- Train loss: 0.31408095359802246, Train Acc: 0.94921875 Test Acc: 0.9443359375
Step 31 -- Train loss: 0.3320361375808716, Train Acc: 0.94482421875 Test Acc: 0.943359375
Step 32 -- Train loss: 0.32411816716194153, Train Acc: 0.94580078125 Test Acc: 0.9501953125
Step 33 -- Train loss: 0.3333812654018402, Train Acc: 0.94482421875 Test Acc: 0.9443359375
Step 34 -- Train loss: 0.3221978545188904, Train Acc: 0.94677734375 Test Acc: 0.9560546875
Step 35 -- Tr

: 

In [4]:
import torch
import argparse
import math

def load_state_dict(path):
    ckpt = torch.load(path, map_location="cpu")

    # Case 1: saved via torch.save(model.state_dict())
    if isinstance(ckpt, dict) and all(isinstance(v, torch.Tensor) for v in ckpt.values()):
        return ckpt

    # Case 2: training checkpoint with 'state_dict'
    if "state_dict" in ckpt:
        return ckpt["state_dict"]

    # Case 3: sometimes saved under 'model'
    if "model" in ckpt:
        return ckpt["model"]

    raise ValueError("Could not find state_dict in checkpoint.")

def relative_diff(a, b, eps=1e-12):
    # Stable relative difference:
    # |a - b| / (|b| + eps)
    return torch.abs(a - b) / (torch.abs(b) + eps)

def compare_checkpoints(path1, path2):
    sd1 = load_state_dict(path1)
    sd2 = load_state_dict(path2)

    keys1 = set(sd1.keys())
    keys2 = set(sd2.keys())

    print("\n=== Key Comparison ===")
    print("Only in ckpt1:", keys1 - keys2)
    print("Only in ckpt2:", keys2 - keys1)

    shared_keys = sorted(keys1 & keys2)

    print(f"\nComparing {len(shared_keys)} shared layers...\n")

    results = []

    for k in shared_keys:
        w1 = sd1[k]
        w2 = sd2[k]

        if w1.shape != w2.shape:
            print(f"Shape mismatch for {k}: {w1.shape} vs {w2.shape}")
            continue

        diff = w1 - w2
        abs_diff = torch.abs(diff)

        max_abs = abs_diff.max().item()
        mean_abs = abs_diff.mean().item()
        l2_diff = torch.norm(diff).item()

        # Relative difference
        rel = relative_diff(w1, w2)
        mean_rel = rel.mean().item()
        max_rel = rel.max().item()

        # Cosine similarity (flatten first)
        cos = torch.nn.functional.cosine_similarity(
            w1.flatten(), w2.flatten(), dim=0
        ).item()

        results.append((k, max_abs, mean_abs, l2_diff, mean_rel, max_rel, cos))

    # Sort by mean absolute diff (largest change first)
    results.sort(key=lambda x: x[2], reverse=True)

    print("=== Layer-wise Differences ===")
    for r in results:
        print(f"\nLayer: {r[0]}")
        print(f"  max_abs_diff   : {r[1]:.6e}")
        print(f"  mean_abs_diff  : {r[2]:.6e}")
        print(f"  l2_diff        : {r[3]:.6e}")
        print(f"  mean_rel_diff  : {r[4]:.6e}")
        print(f"  max_rel_diff   : {r[5]:.6e}")
        print(f"  cosine_sim     : {r[6]:.6f}")

    print("\nDone.")

compare_checkpoints('ckpt_after_prefix.pt', 'ckpt_after_mws.pt')


=== Key Comparison ===
Only in ckpt1: set()
Only in ckpt2: set()

Comparing 22 shared layers...

=== Layer-wise Differences ===

Layer: transformer.h.0.ln_2.weight
  max_abs_diff   : 1.279759e-02
  mean_abs_diff  : 6.330658e-03
  l2_diff        : 1.106772e-01
  mean_rel_diff  : 6.183187e-03
  max_rel_diff   : 1.241545e-02
  cosine_sim     : 0.999996

Layer: transformer.ln_f.weight
  max_abs_diff   : 7.815957e-03
  mean_abs_diff  : 4.830150e-03
  l2_diff        : 7.885186e-02
  mean_rel_diff  : 4.660342e-03
  max_rel_diff   : 7.492793e-03
  cosine_sim     : 1.000000

Layer: transformer.ln_f.bias
  max_abs_diff   : 7.388988e-03
  mean_abs_diff  : 3.267570e-03
  l2_diff        : 5.941263e-02
  mean_rel_diff  : 3.613085e-01
  max_rel_diff   : 6.912502e+00
  cosine_sim     : 0.997189

Layer: transformer.wpe.weight
  max_abs_diff   : 1.007245e-02
  mean_abs_diff  : 3.095859e-03
  l2_diff        : 3.335173e-01
  mean_rel_diff  : 1.172900e+00
  max_rel_diff   : 1.352100e+03
  cosine_sim     :